[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb)

# 🟡 Medium: BatchNorm の実装

**Batch Normalization** を **training** と **inference** 両モードで実装する。

training mode では **batch 統計量** を使い、running estimate を更新：

$$\text{BN}(x) = \gamma \cdot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$$

ここで $\mu_B$ と $\sigma_B^2$ は **batch にまたがる** mean と variance (dim=0)。

inference mode では現在の batch stats じゃなく、与えられた **running mean/var** を使う。

### Signature
```python
def my_batch_norm(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
) -> torch.Tensor:
    # x: (N, D) — batch の全 sample にまたがって各 feature を normalize
    # running_mean, running_var: training 中は in-place で更新、inference ではそのまま使用
```

### Rules
- `F.batch_norm`, `nn.BatchNorm1d` などは **使わない**
- batch mean/variance は `dim=0` で `unbiased=False` で計算
- running stats の更新は PyTorch 流: `running = (1 - momentum) * running + momentum * batch_stat`
- `training=False` の時は `running_mean` / `running_var` を使う
- `x`, `gamma`, `beta` についての autograd をサポート（running stats は buffer 扱いで勾配不要）

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_batch_norm(
    x,
    gamma,
    beta,
    running_mean,
    running_var,
    eps=1e-5,
    momentum=0.1,
    training=True,
):
    pass  # training: batch stats で正規化 + running stats を in-place 更新
          #         inference: running stats で正規化


In [ ]:
# 🧪 Debug
x = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

# Running stats typically live on the same device and shape as features
running_mean = torch.zeros(4)
running_var = torch.ones(4)

# Training mode: uses batch stats and updates running_mean / running_var
out_train = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
print("[Train] Output shape:", out_train.shape)
print("[Train] Column means:", out_train.mean(dim=0))   # should be ~0
print("[Train] Column stds: ", out_train.std(dim=0))    # should be ~1
print("Updated running_mean:", running_mean)
print("Updated running_var:", running_var)

# Inference mode: uses running_mean / running_var only
out_eval = my_batch_norm(x, gamma, beta, running_mean, running_var, training=False)
print("[Eval] Output shape:", out_eval.shape)

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("batchnorm")